# Phase 1 Final Controls Metric Formula Revision Plan

Claim-closed orchestration notebook. This notebook records the revision decision boundary after metric-contract ambiguity is detected. It does not select a formula, change thresholds, edit logits or metrics, rerun controls, or open Phase 1 claims.

If the upstream metric-contract audit already locks the runtime formula and closes ambiguity, this notebook must remain claim-closed and stop the manual revision chain rather than reopening a resolved contract question.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private, hoac de trong neu public: ')
    if token:
        header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
if RUN_UNITTESTS:
    unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
    if unit_result.returncode != 0:
        raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed metric-contract audit source and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Preferred: pin reviewed source explicitly. Leave as None to resolve latest valid run.
PINNED_METRIC_CONTRACT_AUDIT_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_controls_metric_contract_audit/20260423T170705285760Z')

def latest_run_with_file(root: Path, required_file: str) -> Path:
    candidates = sorted([
        p for p in root.iterdir()
        if p.is_dir() and (p / required_file).exists()
    ])
    if not candidates:
        raise FileNotFoundError(f'No runs with {required_file} under {root}')
    print(f'Found runs under {root}:', len(candidates))
    for p in candidates[-5:]:
        print(p, 'required file exists =', (p / required_file).exists())
    return candidates[-1]

if PINNED_METRIC_CONTRACT_AUDIT_RUN is None:
    METRIC_CONTRACT_AUDIT_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_controls_metric_contract_audit',
        'phase1_final_controls_metric_contract_audit_summary.json',
    )
else:
    METRIC_CONTRACT_AUDIT_RUN = Path(PINNED_METRIC_CONTRACT_AUDIT_RUN)

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_revision_plan'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_revision_plan_manual_hold'

RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN = False
REQUIRED_ACK = 'I reviewed final controls metric formula revision planning and understand it selects no formula changes no thresholds and opens no claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_controls_metric_formula_revision_plan_v2_locked_formula_guard_20260423'

print('Notebook fix marker:', FIX_MARKER)
print('Metric-contract audit run:', METRIC_CONTRACT_AUDIT_RUN)
print('Metric-contract summary exists:', (METRIC_CONTRACT_AUDIT_RUN / 'phase1_final_controls_metric_contract_audit_summary.json').exists())
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN)


In [ ]:
# Cell 3 - Utilities and prereg/source validation.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_prereg_bundle(path: Path) -> Path:
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Found prereg bundles:', len(candidates))
    for candidate in candidates:
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_IDENTITY_HASH:
            return candidate
    raise FileNotFoundError(f'No locked prereg bundle found for expected identity hash under {ARTIFACT_ROOT / "prereg"}')

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
METRIC_CONTRACT_AUDIT_RUN = Path(METRIC_CONTRACT_AUDIT_RUN)

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

metric_summary = read_json(METRIC_CONTRACT_AUDIT_RUN / 'phase1_final_controls_metric_contract_audit_summary.json')
metric_claim_state = read_json(METRIC_CONTRACT_AUDIT_RUN / 'phase1_final_controls_metric_contract_claim_state.json')
metric_formula_contract_resolved = (
    metric_summary.get('relative_formula_locked') is True
    and metric_summary.get('formula_ambiguity_detected') is False
)
metric_formula_revision_still_needed = (
    metric_summary.get('relative_formula_locked') is False
    and metric_summary.get('formula_ambiguity_detected') is True
)

assert metric_summary.get('claims_opened') is False
assert metric_summary.get('claim_ready') is False
assert metric_claim_state.get('headline_phase1_claim_open') is False
assert metric_formula_contract_resolved or metric_formula_revision_still_needed, (
    'Unexpected metric-contract state for revision planning. Review the audit summary before proceeding.'
)

print('Formula ambiguity detected:', metric_summary.get('formula_ambiguity_detected'))
print('Relative formula locked:', metric_summary.get('relative_formula_locked'))
print('Controls with formula-dependent pass status:', metric_summary.get('controls_with_formula_dependent_pass_status'))
print('Runtime formula ids:', metric_summary.get('current_runtime_formula_ids'))
print('Metric formula contract resolved upstream:', metric_formula_contract_resolved)
if metric_formula_contract_resolved:
    print('NOTE: Upstream metric-contract audit already locked the runtime formula. This notebook should stay held and the 38-40 remediation chain is not needed for this ambiguity issue.')

preflight = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_controls_metric_formula_revision_plan', '--help'],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)
runner_available = preflight.returncode == 0
print('Runner available:', runner_available)
assert runner_available, preflight.stderr


In [ ]:
# Cell 4 - Record manual-hold plan and exact CLI command.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / created_utc
cmd = [
    'python', '-m', 'src.cli', 'phase1_final_controls_metric_formula_revision_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--metric-contract-audit-run', str(METRIC_CONTRACT_AUDIT_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--revision-plan-config', 'configs/phase1/final_controls_metric_formula_revision_plan.json',
    '--metric-contract-config', 'configs/phase1/final_controls_metric_contract_audit.json',
    '--gate2-config', 'configs/gate2/synthetic_validation.json',
]
plan = {
    'status': 'phase1_final_controls_metric_formula_revision_plan_manual_hold_recorded',
    'created_utc': created_utc,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'metric_contract_audit_run': str(METRIC_CONTRACT_AUDIT_RUN),
    'source_formula_ambiguity_detected': metric_summary.get('formula_ambiguity_detected'),
    'source_relative_formula_locked': metric_summary.get('relative_formula_locked'),
    'source_metric_formula_contract_resolved': metric_formula_contract_resolved,
    'source_controls_in_scope': metric_summary.get('controls_with_formula_dependent_pass_status'),
    'command': cmd,
    'scientific_limit': 'This plan records only a claim-closed formula revision decision boundary; it selects no formula.',
}
write_json(plan_dir / 'phase1_final_controls_metric_formula_revision_plan_manual_hold.json', plan)
print('Plan source of truth:', plan_dir)
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual launch gate.

if not RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN:
    hold = {
        'status': 'phase1_final_controls_metric_formula_revision_plan_manual_hold',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'run_flag': RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
        'metric_formula_contract_resolved_upstream': metric_formula_contract_resolved,
        'claims_opened': False,
    }
    print(json.dumps(hold, indent=2))
elif metric_formula_contract_resolved:
    raise RuntimeError('Revision-plan launch not allowed: upstream metric-contract audit already locked the runtime formula and closed ambiguity.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch formula revision planning without explicit claim-closed acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_controls_metric_formula_revision_plan_launch_acknowledged',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'ack_matched': True,
        'claims_opened': False,
    }
    write_json(plan_dir / 'phase1_final_controls_metric_formula_revision_plan_launch_review.json', launch_review)
    run(cmd, cwd=REPO_DIR)
    print('Metric-formula revision plan command completed. Review the revision decision memo before any formula choice.')


In [ ]:
# Cell 6 - Inspect latest formula revision-plan output.

latest_txt = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest_txt.exists())
if latest_txt.exists():
    latest_run = Path(latest_txt.read_text(encoding='utf-8').strip())
else:
    runs = sorted([path for path in OUTPUT_ROOT.iterdir() if path.is_dir()]) if OUTPUT_ROOT.exists() else []
    latest_run = runs[-1] if runs else None

if latest_run is None:
    print('No formula revision-plan output yet; this is expected for manual hold.')
else:
    print('Latest formula revision-plan output:', latest_run)
    expected_files = [
        'phase1_final_controls_metric_formula_revision_plan_inputs.json',
        'phase1_final_controls_metric_formula_revision_plan_summary.json',
        'phase1_final_controls_metric_formula_revision_plan_report.md',
        'phase1_final_controls_metric_formula_revision_source_links.json',
        'phase1_final_controls_metric_formula_revision_input_validation.json',
        'phase1_final_controls_metric_formula_options.json',
        'phase1_final_controls_metric_formula_revision_scope.json',
        'phase1_final_controls_metric_formula_decision_requirements.json',
        'phase1_final_controls_metric_formula_revision_claim_state.json',
        'phase1_final_controls_metric_formula_revision_decision_memo.md',
    ]
    for name in expected_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_controls_metric_formula_revision_plan_summary.json')
    scope = read_json(latest_run / 'phase1_final_controls_metric_formula_revision_scope.json')
    requirements = read_json(latest_run / 'phase1_final_controls_metric_formula_decision_requirements.json')
    print(json.dumps({
        'status': summary.get('status'),
        'claim_ready': summary.get('claim_ready'),
        'claims_opened': summary.get('claims_opened'),
        'revision_required': summary.get('revision_required'),
        'manual_decision_required': summary.get('manual_decision_required'),
        'controls_in_scope': summary.get('controls_in_scope'),
        'runtime_formula_ids': summary.get('runtime_formula_ids'),
        'selected_formula': summary.get('selected_formula'),
        'code_change_allowed_now': summary.get('code_change_allowed_now'),
        'rerun_controls_allowed_now': summary.get('rerun_controls_allowed_now'),
        'threshold_change_allowed_now': summary.get('threshold_change_allowed_now'),
        'next_step': summary.get('next_step'),
        'scientific_limit': summary.get('scientific_limit'),
    }, indent=2))


In [ ]:
# Cell 7 - Assertions and local review note.

if RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN:
    assert latest_run is not None, 'Expected formula revision-plan output after launch.'
    summary = read_json(latest_run / 'phase1_final_controls_metric_formula_revision_plan_summary.json')
    options = read_json(latest_run / 'phase1_final_controls_metric_formula_options.json')
    scope = read_json(latest_run / 'phase1_final_controls_metric_formula_revision_scope.json')
    requirements = read_json(latest_run / 'phase1_final_controls_metric_formula_decision_requirements.json')
    claim_state = read_json(latest_run / 'phase1_final_controls_metric_formula_revision_claim_state.json')

    assert summary.get('status') in {'phase1_final_controls_metric_formula_revision_plan_recorded', 'phase1_final_controls_metric_formula_revision_plan_blocked'}, summary
    assert summary.get('claim_ready') is False
    assert summary.get('claims_opened') is False
    assert summary.get('selected_formula') is None
    assert summary.get('code_change_allowed_now') is False
    assert summary.get('rerun_controls_allowed_now') is False
    assert summary.get('threshold_change_allowed_now') is False
    assert options.get('selected_formula') is None
    assert scope.get('revision_scope') == 'metric_formula_contract_only'
    assert requirements.get('decision_must_be_made_before_code_change') is True
    assert claim_state.get('headline_phase1_claim_open') is False

    review_note = {
        'status': 'phase1_final_controls_metric_formula_revision_plan_review_pass_claim_closed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_run),
        'checks_passed': [
            'required_artifacts_present',
            'claim_ready_false',
            'claims_opened_false',
            'selected_formula_none',
            'code_change_not_allowed_yet',
            'rerun_controls_not_allowed_yet',
            'threshold_change_not_allowed',
            'claim_state_closed',
        ],
        'revision_required': summary.get('revision_required'),
        'manual_decision_required': summary.get('manual_decision_required'),
        'controls_in_scope': summary.get('controls_in_scope'),
        'runtime_formula_ids': summary.get('runtime_formula_ids'),
        'relative_formula_locked_before_revision': scope.get('relative_formula_locked_before_revision'),
        'next_step': summary.get('next_step'),
        'allowed_interpretation': 'Revision-planning artifact only. If the source metric contract is already locked, this notebook documents that no manual formula revision decision is needed for the resolved ambiguity.',
        'not_allowed_interpretation': 'Do not treat this plan as a formula choice, code change authorization, control pass, or Phase 1 efficacy evidence.',
        'not_ok_to_claim': claim_state.get('not_ok_to_claim', []),
    }
    if summary.get('revision_required') is True:
        review_note['checks_passed'].extend([
            'revision_required_true',
            'manual_decision_required_true',
        ])
    else:
        review_note['checks_passed'].extend([
            'revision_not_required_after_formula_lock',
            'manual_decision_not_required',
        ])
    write_json(latest_run / 'phase1_final_controls_metric_formula_revision_plan_review_note.json', review_note)
    print('Review note written:', latest_run / 'phase1_final_controls_metric_formula_revision_plan_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch flag is False. This is expected during first-pass manual hold.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Final Controls Metric Formula Revision Plan Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Metric-contract audit run:', METRIC_CONTRACT_AUDIT_RUN)
print('Source formula ambiguity detected:', metric_summary.get('formula_ambiguity_detected'))
print('Source relative formula locked:', metric_summary.get('relative_formula_locked'))
print('Metric formula contract resolved upstream:', metric_formula_contract_resolved)

if not RUN_FINAL_CONTROLS_METRIC_FORMULA_REVISION_PLAN:
    if metric_formula_contract_resolved:
        print('HELD: Source metric contract is already locked and ambiguity is closed, so this revision-plan notebook is not needed for the current issue.')
        print('NEXT: stop the formula-remediation chain here; do not run notebooks 38-40 for this resolved contract ambiguity.')
    else:
        print('HELD: Formula revision plan was not launched because manual flag is False.')
        print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
else:
    print('Latest formula revision-plan output:', latest_run)
    print('Revision required:', summary.get('revision_required'))
    print('Manual decision required:', summary.get('manual_decision_required'))
    print('Controls in scope:', summary.get('controls_in_scope'))
    print('Runtime formula ids:', summary.get('runtime_formula_ids'))
    print('Selected formula:', summary.get('selected_formula'))
    print('Code change allowed now:', summary.get('code_change_allowed_now'))
    print('Rerun controls allowed now:', summary.get('rerun_controls_allowed_now'))
    print('Threshold change allowed now:', summary.get('threshold_change_allowed_now'))
    print('Next step:', summary.get('next_step'))
    if summary.get('revision_required') is True:
        print('CHECK REQUIRED: Review phase1_final_controls_metric_formula_revision_decision_memo.md before choosing any formula or editing code/config.')
    else:
        print('CHECK REQUIRED: Review phase1_final_controls_metric_formula_revision_decision_memo.md, then stop the 38-40 revision chain unless a new contract discrepancy is discovered.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
